# <center> VAI Store - Engenharia de Variáveis </center>

---

Este notebook implementa o pipeline final de engenharia de variáveis. Com base nas análises de modelo e nos requisitos do case, esta abordagem é desenhada para:

1. Prever a demanda ao nível do SKU, o que é necessário para responder às perguntas de negócio.

2. Usar o FATUR_TOTAL (Receita) como a métrica-alvo, resolvendo o problema de unidades de medida incomparáveis (ex: KG vs. UN).

3. Criar features de lag normalizadas e features de interação de eventos (ex: Páscoa + SKU 1813) para capturar a sazonalidade e os picos de venda específicos.

### 1. Configuração e Imports

Visão Geral: Esta etapa importa as bibliotecas necessárias para a manipulação de dados (pandas, numpy), gerenciamento de arquivos (os) e criação de features de calendário (holidays). Os caminhos relativos (../) são configurados para ler os dados da pasta data/raw e salvar o dataset final em data/processed.

In [16]:
import pandas as pd
import numpy as np
import os
import holidays

# Define os caminhos relativos (subindo um nível)
RAW_DATA_PATH = os.path.join('..', 'data', 'raw')
VENDAS_FILE = os.path.join(RAW_DATA_PATH, 'vendas.csv')
PRODUTO_FILE = os.path.join(RAW_DATA_PATH, 'produto.csv')

PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed')
# Arquivo de saída final, agregado por SKU
OUTPUT_FILE = os.path.join(PROCESSED_DATA_PATH, 'dataset.parquet')

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

### 2. Funções Auxiliares de Carga e Limpeza

Visão Geral: São definidas funções auxiliares para manter o código limpo e reutilizável. A função load_csv trata de possíveis erros de codificação (utf-8 ou latin1) na leitura dos arquivos. A função clean_sku padroniza a coluna SKU, removendo espaços e convertendo-a para um tipo numérico, garantindo a integridade da chave de junção.

In [17]:
def clean_sku(sku_series):
    """Limpa a coluna SKU, removendo espaços e convertendo para inteiro."""
    return pd.to_numeric(sku_series.astype(str).str.strip(), errors='coerce').astype('Int64')

def load_csv(file_path):
    """Tenta carregar um CSV com encoding 'utf-8', e usa 'latin1' como fallback."""
    try:
        return pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='latin1')

### 3. Carregar e Agregar Dados de Vendas

Visão Geral: Esta etapa carrega os dados brutos de vendas.csv. Os dados são mantidos no nível transacional (linha por linha de venda). As colunas DATA_ATEND e SKU passam por uma limpeza inicial. Manter os dados desagregados nesta fase é crucial para que, na próxima etapa, a CATEGORIA de cada produto possa ser associada a cada venda individual.

In [18]:
df_vendas = load_csv(VENDAS_FILE)

# Limpezas básicas
df_vendas['DATA_ATEND'] = pd.to_datetime(df_vendas['DATA_ATEND'])
df_vendas['SKU'] = clean_sku(df_vendas['SKU'])

# Remove SKUs nulos e vendas com faturamento zero (não contribuem)
df_vendas = df_vendas.dropna(subset=['SKU'])
df_vendas = df_vendas[df_vendas['FATUR_VENDA'] > 0]

### 4. Adicionar Contexto do Produto

Visão Geral: Esta etapa carrega os dados de produto.csv. O objetivo é criar um "mapa" limpo que associa cada SKU ao seu NOME_PRODUTO, CATEGORIA e SUBCATEGORIA. Os valores ausentes (NaN) nas colunas de categoria são explicitamente preenchidos com "DESCONHECIDA", garantindo que nenhum SKU seja perdido em etapas futuras de junção.

In [19]:
print(f"Carregando dados de contexto de {PRODUTO_FILE}...")

# Carrega e limpa os dados de produtos
df_prod = load_csv(PRODUTO_FILE)
df_prod['SKU'] = clean_sku(df_prod['SKU'])
df_prod = df_prod.dropna(subset=['SKU'])

# Seleciona, limpa e padroniza as colunas de contexto
df_prod_context = df_prod[['SKU', 'NOME_PRODUTO', 'CATEGORIA', 'SUBCATEGORIA']].copy()
df_prod_context['CATEGORIA'] = df_prod_context['CATEGORIA'].fillna('DESCONHECIDA')
df_prod_context['SUBCATEGORIA'] = df_prod_context['SUBCATEGORIA'].fillna('DESCONHECIDA')
df_prod_context['NOME_PRODUTO'] = df_prod_context['NOME_PRODUTO'].fillna('NOME DESCONHECIDO')

# Garante uma chave única
df_prod_context = df_prod_context.drop_duplicates(subset=['SKU'])

Carregando dados de contexto de ..\data\raw\produto.csv...


### 5. Juntar Contexto (Nível Transacional)

Visão Geral: Esta etapa enriquece os dados transacionais de vendas. O "mapa" de contexto (da Etapa 4) é mesclado ao dataset de vendas (da Etapa 3), usando o SKU como chave. O resultado é que cada linha de transação agora "sabe" a sua respetiva CATEGORIA e SUBCATEGORIA, o que é essencial para as features de interação.

In [20]:
# Mescla o contexto ao dataset transacional de vendas
df_vendas_com_contexto = pd.merge(df_vendas, df_prod_context, on='SKU', how='left')

# Preenche 'DESCONHECIDA' para SKUs de vendas que não estavam no 'produto.csv'
df_vendas_com_contexto['CATEGORIA'] = df_vendas_com_contexto['CATEGORIA'].fillna('DESCONHECIDA')
df_vendas_com_contexto['SUBCATEGORIA'] = df_vendas_com_contexto['SUBCATEGORIA'].fillna('DESCONHECIDA')

### 6. Agregar Faturamento por SKU

Visão Geral: Esta é a etapa central de agregação. Os dados transacionais (agora com contexto) são agregados ao nível de DATA_ATEND e SKU. O nosso alvo, FATUR_TOTAL, é calculado somando o FATUR_VENDA. Este dataset agregado "esparso" (contendo 21.535 linhas, como validado) forma a base para o nosso scaffold.

In [21]:
# Usamos df_vendas_com_contexto para garantir que só SKUs com vendas sejam agregados
df_agg_sku = df_vendas_com_contexto.groupby(['DATA_ATEND', 'SKU']).agg(
    FATUR_TOTAL=('FATUR_VENDA', 'sum')
).reset_index()

### 7. Criar "Scaffold" (Grid de Datas Completo)

Visão Geral: Os dados agregados por SKU (da Etapa 6) são "esparsos" (só têm linhas em dias de venda). Para o modelo aprender com os dias de faturamento zero, esta etapa cria um "scaffold" (gabarito denso). O scaffold contém uma linha para cada dia (forçando o início em 1º de Janeiro) e para cada SKU que já teve pelo menos uma venda. Os dados de faturamento agregados são, então, mesclados a este gabarito.

In [22]:
group_cols = ['SKU']

# --- Forçar o início em 1º de Janeiro ---
# Encontra o ano mínimo e a data máxima dos dados de vendas
min_year = df_agg_sku['DATA_ATEND'].min().year
max_date = df_agg_sku['DATA_ATEND'].max()

# Força o min_date a ser 1º de Janeiro daquele ano
min_date = pd.to_datetime(f'{min_year}-01-01')
# --- Fim da Correção ---

date_range = pd.date_range(min_date, max_date, freq='D')

# Encontra os SKUs únicos (APENAS os que tiveram vendas, 59 SKUs)
unique_skus = df_agg_sku['SKU'].unique()

# Cria o scaffold
df_scaffold = pd.MultiIndex.from_product(
    [unique_skus, date_range],
    names=['SKU', 'DATA_ATEND']
).to_frame(index=False)

# Junta os dados reais (faturamento) ao scaffold
# O df_full terá agora (59 SKUs * 366 Dias) = 21.594 linhas
df_full = pd.merge(df_scaffold, df_agg_sku, on=['SKU', 'DATA_ATEND'], how='left')

### 8. Preencher Dados Ausentes e Adicionar Contexto

Visão Geral: Esta etapa finaliza o dataset denso.

* Preenchimento de Zeros: O FATUR_TOTAL (que está NaN nos dias de não-venda) é preenchido com 0.

* Junção de Contexto: O "mapa" de contexto (SKU -> Categoria, da Etapa 4) é mesclado ao dataset denso (df_full). Esta abordagem robusta garante que o dataset final mantenha tanto os zeros de faturamento quanto o contexto do produto.

In [23]:
# 1. Preenche métricas (faturamento) com 0
df_full['FATUR_TOTAL'] = df_full['FATUR_TOTAL'].fillna(0)

# 2. Usa o mapa de contexto limpo da Etapa 4
# (df_prod_context já tem SKUs únicos e NaNs preenchidos)

# 3. Junta o contexto ao df_full (scaffold)
df_full = pd.merge(df_full, df_prod_context, on='SKU', how='left')

# 4. Garante que nenhum contexto seja nulo (fallback)
df_full['CATEGORIA'] = df_full['CATEGORIA'].fillna('DESCONHECIDA')
df_full['SUBCATEGORIA'] = df_full['SUBCATEGORIA'].fillna('DESCONHECIDA')

print(f"Preenchimento e junção de contexto concluídos. Shape: {df_full.shape}")

Preenchimento e junção de contexto concluídos. Shape: (21594, 6)


### 9. Engenharia de Features de Calendário

Visão Geral: Para capturar a sazonalidade, o modelo precisa de features numéricas que representem o tempo. Esta etapa extrai diversas features da DATA_ATEND, incluindo:

* Ciclos Curtos: DIA_SEMANA, DIA_DO_MES.

* Ciclos Longos: DIA_DO_ANO, MES, ANO, SEMANA_DO_ANO.

* Eventos de Negócio: INICIO_MES e FIM_MES (capturando os picos de "dia de pagamento").

In [24]:
date_col = df_full['DATA_ATEND']

df_full['DIA_SEMANA'] = date_col.dt.dayofweek
df_full['DIA_DO_MES'] = date_col.dt.day
df_full['DIA_DO_ANO'] = date_col.dt.dayofyear
df_full['MES'] = date_col.dt.month
df_full['SEMANA_DO_ANO'] = date_col.dt.isocalendar().week.astype(int)
df_full['ANO'] = date_col.dt.year

df_full['INICIO_MES'] = (date_col.dt.day <= 5).astype(int)
df_full['FIM_MES'] = (date_col.dt.day >= 28).astype(int)

### 10. Engenharia de Features de Feriados e Eventos

Visão Geral: Com base na análise exploratória (que mostrou picos em feriados), esta etapa codifica os eventos sazonais mais importantes. Usando a biblioteca holidays, são criadas flags binárias para EH_FERIADO e janelas de antecipação para os principais eventos de vendas: ANTECIPACAO_PASCOA_14D, ANTECIPACAO_NATAL_21D e ANTECIPACAO_ANO_NOVO_7D.

In [25]:
anos = df_full['ANO'].unique()
br_holidays = holidays.Brazil(years=anos)

# Páscoa por ano (dict ano -> data)
pascoa_datas = {}
for dt, name in br_holidays.items():
    if name == 'Páscoa':
        pascoa_datas[pd.to_datetime(dt).year] = pd.to_datetime(dt)

# Natal por ano
natal_datas = {int(ano): pd.to_datetime(f"{int(ano)}-12-25") for ano in anos}

# Mapeia datas por linha
data_atend = pd.to_datetime(df_full['DATA_ATEND'])
df_full['DATA_PASCOA'] = df_full['ANO'].map(pascoa_datas)
df_full['DATA_NATAL']  = df_full['ANO'].map(natal_datas)

# Deltas (dias) para eventos
dias_ate_pascoa = (pd.to_datetime(df_full['DATA_PASCOA']) - data_atend).dt.days
dias_ate_natal  = (pd.to_datetime(df_full['DATA_NATAL'])  - data_atend).dt.days
dias_desde_natal = (data_atend - pd.to_datetime(df_full['DATA_NATAL'])).dt.days

# Se houver NOME_PRODUTO usa o texto; caso contrário usa CATEGORIA
if 'NOME_PRODUTO' in df_full.columns:
    is_bac = df_full['NOME_PRODUTO'].astype(str).str.upper().str.contains('BACALHAU|PESC', na=False)
else:
    is_bac = df_full['CATEGORIA'].astype(str).str.upper().str.contains('BACALHAU|PESC', na=False)

TOP_BACALHAU = {1813, 6159, 3449, 3641, 2033}
is_top = df_full['SKU'].astype(str).isin({str(x) for x in TOP_BACALHAU})

# Compacta duas flags em um único tier (0=outros, 1=Bacalhau não-top, 2=Bacalhau top)
df_full['bacalhau_tier'] = (
    2*is_top.astype(int) + (is_bac.astype(int) & (~is_top).astype(bool)).astype(int)
).astype('int8')

# Pré-Natal (apenas antes do evento; janela até 60d)
pre_nat = np.where((dias_ate_natal >= 0) & (dias_ate_natal <= 60),
                   np.exp(-0.08 * dias_ate_natal.clip(0, 60)), 0.0)

# Pré-Páscoa (apenas antes do evento; janela até 45d)
pre_pas = np.where((dias_ate_pascoa >= 0) & (dias_ate_pascoa <= 45),
                   np.exp(-0.10 * dias_ate_pascoa.clip(0, 45)), 0.0)

# Pico curto de meados de janeiro (âncora 20/jan; janela ±10d; útil para redes com esse padrão)
data_jan20 = pd.to_datetime(df_full['ANO'].astype(int).astype(str) + "-01-20")
dias_ate_jan20 = (data_jan20 - data_atend).dt.days
jan20 = np.exp(-0.22 * np.abs(dias_ate_jan20.clip(-10, 10)))

# Composição (peso igual; aplicado apenas a Bacalhau via tier>0)
evento_forca_raw = np.maximum(pre_nat, pre_pas)
evento_forca_raw = np.maximum(evento_forca_raw, jan20)
df_full['bacalhau_evento_forca'] = (
    evento_forca_raw * (df_full['bacalhau_tier'] > 0).astype(int)
).astype('float32')

df_full['bacalhau_pos_natal'] = (
    np.where(dias_desde_natal > 0, np.clip(dias_desde_natal, 0, 40) / 40.0, 0.0) *
    (df_full['bacalhau_tier'] > 0).astype(int)
).astype('float32')

drop_cols = [c for c in ['DATA_PASCOA','DATA_NATAL'] if c in df_full.columns]
if drop_cols:
    df_full = df_full.drop(columns=drop_cols)

for old in ['IS_BACALHAU','is_top_bacalhau','intensidade_pascoa_bacalhau',
            'intensidade_natal_bacalhau','dias_desde_natal','intensidade_jan20_bacalhau',
            'DIAS_ATE_PASCOA','DIAS_ATE_NATAL','DIAS_ATE_ANO_NOVO',
            'ANTECIPACAO_PASCOA_14D','ANTECIPACAO_NATAL_21D','ANTECIPACAO_ANO_NOVO_7D','EH_FERIADO']:
    if old in df_full.columns:
        df_full = df_full.drop(columns=[old])

df_full['bacalhau_tier'] = df_full['bacalhau_tier'].astype('int8')


### 11. Engenharia de Features de Lag (Memória)

Visão Geral: Para dar "memória" ao modelo, são criadas features de lag (defasagem). O foco é no histórico de curto prazo (1, 2 e 3 dias atrás) e na sazonalidade semanal (7 e 14 dias atrás). O alvo para os lags é o FATUR_TOTAL e o agrupamento é feito por SKU. O df_full (denso, com 21.594 linhas) garante que os lags reflitam corretamente os dias de faturamento zero.

In [26]:
# Define os lags de curto prazo e sazonais
lags = [1, 2, 3, 7, 14]
target_col = 'FATUR_TOTAL'
group_cols = ['SKU'] # A agregação agora é por SKU

# Ordena os dados para garantir que o shift seja temporalmente correto
df_full = df_full.sort_values(by=['SKU', 'DATA_ATEND'])

for lag in lags:
    col_name = f'{target_col}_LAG_{lag}'
    df_full[col_name] = df_full.groupby(group_cols)[target_col].shift(lag)


### 12. Normalização das Features de Lag (SKU-Específica)

Visão Geral: A análise de erros anterior mostrou que o modelo confiava excessivamente nos LAGs (ex: R$ 10.000), "abafando" o sinal das features de INTERACAO (que são 0/1). Esta etapa normaliza os lags, dividindo-os pela média histórica de faturamento daquele SKU (calculada apenas em dias de venda). A feature resultante (ex: LAG_1_NORM) mede o faturamento de ontem como um múltiplo da média (ex: "1.2x a média"), forçando o modelo a usar as features de INTERACAO para explicar os picos.

In [27]:
# Para calcular a média "real" de venda, filtramos dias com faturamento > 0
media_sku = df_full[df_full['FATUR_TOTAL'] > 0].groupby('SKU')['FATUR_TOTAL'].mean()
# Adiciona 1 centavo para evitar divisão por zero
media_sku = media_sku.replace(0, 0.01).fillna(0.01)

# 2. Mapear a média de volta ao df_full
df_full['FATUR_MEDIO_SKU'] = df_full['SKU'].map(media_sku)
df_full['FATUR_MEDIO_SKU'] = df_full['FATUR_MEDIO_SKU'].fillna(0.01)

# 3. Definir os lags que queremos normalizar
lags_a_normalizar = [
    'FATUR_TOTAL_LAG_1',
    'FATUR_TOTAL_LAG_2',
    'FATUR_TOTAL_LAG_3',
    'FATUR_TOTAL_LAG_7',
    'FATUR_TOTAL_LAG_14'
]

# 4. Normalizar os Lags (Dividir pela Média do SKU)
for col in lags_a_normalizar:
    col_norm = f'{col}_NORM'
    # Dividimos o LAG pela média do SKU
    df_full[col_norm] = df_full[col] / df_full['FATUR_MEDIO_SKU']
    
    # Removemos a feature de LAG original (absoluta)
    df_full = df_full.drop(columns=[col])

# 5. Limpar a coluna auxiliar
df_full = df_full.drop(columns=['FATUR_MEDIO_SKU'])

### 13. Fourier (mês cíclico)

Visão Geral: Criamos duas variáveis senoidais que representam o mês como ciclo: FOURIER_MES_SIN e FOURIER_MES_COS.
Isso faz o modelo “perceber” que dezembro (12) é vizinho de janeiro (1) no círculo trigonométrico, evitando que trate 1 e 12 como extremos distantes.

In [28]:
# Garante a coluna de data e de mês
df_full['DATA_ATEND'] = pd.to_datetime(df_full['DATA_ATEND'], errors='coerce')
if 'MES' not in df_full.columns:
    df_full['MES'] = df_full['DATA_ATEND'].dt.month

# Fourier de mês (período = 12)
df_full['FOURIER_MES_SIN'] = np.sin(2 * np.pi * df_full['MES'] / 12)
df_full['FOURIER_MES_COS'] = np.cos(2 * np.pi * df_full['MES'] / 12)

# Tratamento simples de NaN (se houver datas faltantes)
df_full[['FOURIER_MES_SIN','FOURIER_MES_COS']] = df_full[['FOURIER_MES_SIN','FOURIER_MES_COS']].fillna(0)


### 14. Loja Fechada

Visão Geral: Criamos a variável loja_fechada com a seguinte regra:
* loja_fechada = 0 quando não houve nenhuma venda no dia (loja fechada)
* loja_fechada = 1 quando houve ao menos uma venda no dia (loja aberta)

In [29]:
# Garante data válida
df_full['DATA_ATEND'] = pd.to_datetime(df_full['DATA_ATEND'], errors='coerce')

# Soma diária de faturamento (todas as linhas/SKUs do dia)
daily_sum = df_full.groupby('DATA_ATEND')['FATUR_TOTAL'].transform('sum')

# loja_fechada: 0 quando não houve nenhuma venda no dia; 1 quando houve
df_full['loja_fechada'] = (daily_sum > 0).astype('int8')

# (Opcional) garantir tipo categórico leve
df_full['loja_fechada'] = df_full['loja_fechada'].astype('int8')

### 15. Limpeza Final e Salvamento

Visão Geral: Esta é a etapa final. O dataset é limpo (preenchendo NaNs iniciais dos lags com 0). O código de otimização de memória é executado, convertendo as features numéricas (_NORM) para float32 e as features de calendário/evento (INTERACAO_) para inteiros menores (int8). O dataset final é salvo em formato Parquet.

In [30]:
cols_to_drop = [c for c in [
    # dummies antigas e marcadores não usados
    'ANTECIPACAO_PASCOA_14D','ANTECIPACAO_NATAL_21D','ANTECIPACAO_ANO_NOVO_7D','EH_FERIADO',
    'IS_BACALHAU','is_top_bacalhau',
    'intensidade_pascoa_bacalhau','intensidade_natal_bacalhau','intensidade_jan20_bacalhau',
    'dias_desde_natal',
    'DIAS_ATE_PASCOA','DIAS_ATE_NATAL','DIAS_ATE_ANO_NOVO'
] if c in df_full.columns]

if cols_to_drop:
    df_full = df_full.drop(columns=cols_to_drop)

required_defaults = [
    ('bacalhau_tier', 0),                 
    ('bacalhau_evento_forca', 0.0),       
    ('bacalhau_pos_natal', 0.0),          
    ('FOURIER_MES_SIN', 0.0),
    ('FOURIER_MES_COS', 0.0),
    ('loja_fechada', 1),                  
    ('FATUR_TOTAL_LAG_1_NORM', 0.0),
    ('FATUR_TOTAL_LAG_2_NORM', 0.0),
    ('FATUR_TOTAL_LAG_3_NORM', 0.0),
    ('FATUR_TOTAL_LAG_7_NORM', 0.0),
]
for col, default in required_defaults:
    if col not in df_full.columns:
        df_full[col] = default

keep_cols = [c for c in [
    'DATA_ATEND','SKU','NOME_PRODUTO',
    'FATUR_TOTAL',
    'DIA_SEMANA',
    'bacalhau_tier','bacalhau_evento_forca','bacalhau_pos_natal',
    'FATUR_TOTAL_LAG_1_NORM','FATUR_TOTAL_LAG_2_NORM','FATUR_TOTAL_LAG_3_NORM','FATUR_TOTAL_LAG_7_NORM',
    'FOURIER_MES_SIN','FOURIER_MES_COS',
    'loja_fechada'
] if c in df_full.columns]

df_full = df_full[keep_cols].copy()

df_final = df_full.fillna(0)

for col in df_final.columns:
    if col == 'FATUR_TOTAL':
        df_final[col] = df_final[col].astype('float32')

    # Lags e Fourier em float32
    if col.endswith('_NORM') or col in ('FOURIER_MES_SIN','FOURIER_MES_COS','bacalhau_evento_forca','bacalhau_pos_natal'):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').astype('float32')

    # Inteiros compactos
    if col in ('DIA_SEMANA','loja_fechada'):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='integer').fillna(0).astype('int8')
    if col == 'bacalhau_tier':
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='integer').fillna(0).astype('int8')

df_final.to_parquet(OUTPUT_FILE, index=False, engine='pyarrow')
df_final.info()


<class 'pandas.core.frame.DataFrame'>
Index: 21594 entries, 0 to 19397
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   DATA_ATEND              21594 non-null  datetime64[ns]
 1   SKU                     21594 non-null  Int64         
 2   NOME_PRODUTO            21594 non-null  object        
 3   FATUR_TOTAL             21594 non-null  float32       
 4   DIA_SEMANA              21594 non-null  int8          
 5   bacalhau_tier           21594 non-null  int8          
 6   bacalhau_evento_forca   21594 non-null  float32       
 7   bacalhau_pos_natal      21594 non-null  float32       
 8   FATUR_TOTAL_LAG_1_NORM  21594 non-null  float32       
 9   FATUR_TOTAL_LAG_2_NORM  21594 non-null  float32       
 10  FATUR_TOTAL_LAG_3_NORM  21594 non-null  float32       
 11  FATUR_TOTAL_LAG_7_NORM  21594 non-null  float32       
 12  FOURIER_MES_SIN         21594 non-null  float32    